# 01 · El método

**Este cuaderno no te enseña a parsear un PDF.** Te enseña dónde poner las
junturas y las defensas, que es lo que hace que una automatización de data entry
siga funcionando el mes que viene.

Son 20 minutos de lectura con algunas celdas que se ejecutan. Después seguís con
`02_construir`, donde lo escribís vos.

---

## Qué vas a construir en este curso

Una automatización que lee datos de un archivo y los carga en un formulario web,
fila por fila, **sin escribir basura en el sistema de destino**.

El curso usa un caso de ejemplo (fichas crediticias de clientes) que viene entero
en el repo, incluida una web falsa de destino. Pero el objetivo no es ese caso:
es que al final de `03_tu_caso` tengas andando **el tuyo**.

## Qué necesitás

| Para el curso | Para tu caso |
|---|---|
| Nada: todo viene en el repo | Un archivo de muestra de tus datos (anonimizado) |
| | Acceso a la web donde cargás hoy a mano |
| | Python 3.10+ en tu máquina |

> **Todo lo que ves acá es inventado.** Los datos, los documentos y la web de
> destino. No hay credenciales ni información real en ningún lado.

## Preparar el entorno

In [ ]:
# Funciona en Colab y en local. Si ya estás dentro del repo, no hace nada.
import os, subprocess, sys
from pathlib import Path

REPO = "https://github.com/GEJ1/data_entry_automatizado.git"

def en_colab():
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False

if not Path("pipeline/contratos.py").exists():
    if Path("data_entry_automatizado/pipeline/contratos.py").exists():
        os.chdir("data_entry_automatizado")
    else:
        print("Clonando el repo...")
        subprocess.run(["git", "clone", "-q", REPO], check=True)
        os.chdir("data_entry_automatizado")

if en_colab():
    print("Instalando dependencias (un par de minutos la primera vez)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    "requirements.txt"], check=True)

PY = sys.executable

def correr(comando, mostrar=True):
    """Corre un comando del proyecto y muestra su salida."""
    r = subprocess.run(comando, shell=True, capture_output=True, text=True)
    salida = (r.stdout + r.stderr).rstrip()
    if mostrar:
        print(salida)
    return salida

def cli(args, mostrar=True):
    return correr(f"{PY} -m pipeline.cli {args}", mostrar)

print("\nlisto ·", Path.cwd(), "· python", sys.version.split()[0])

## El problema, en concreto

Vamos a generar el documento de ejemplo. Es un reporte con fichas de clientes:
una ficha por cliente, dos tablas por ficha.

In [ ]:
correr(f"{PY} demo/generar_pdf_fake.py")

Ese manifiesto no es decorativo: son **once trampas sembradas a propósito**, una
por cada cosa que sale mal en la vida real. Vamos a ir contra ellas una por una.

Ahora mirá el volumen real:

In [ ]:
correr(f"{PY} demo/generar_pdf_fake.py --clientes 300 --salida /tmp/lote300.pdf | head -2")

import pdfplumber
with pdfplumber.open("/tmp/lote300.pdf") as pdf:
    print(f"\n{len(pdf.pages)} páginas · 300 clientes · 2 tablas cada uno")
    print("Alguien copia esto a mano, campo por campo.")

## La pregunta que ordena todo el curso

> Si el programa se equivoca, **¿cómo te enterás?**

Un humano cansado se equivoca al azar: una fila mal cada tanto. Un programa
comete **el mismo error 300 veces**, rapidísimo y sin dudar.

Por eso el trabajo no es escribir el parser. Claude te escribe el parser en dos
minutos. El trabajo es decidir qué pasa cuando dos filas son idénticas, qué
campos no hay que tocar nunca, y cómo verificás que lo que cargaste es lo que
había en el papel.

---

## Las cuatro etapas

```
      PDF ─┐
           ├──[ Extractor ]──→   FichaCruda        texto crudo, sin interpretar
     DOCX ─┘                                       (cabecera + tablas de strings)
                                      │
                                      ▼
                      [ Dominio ]──→   Cliente      ya tipado y validado
                                      │
                                      ▼
                                  lote.jsonl        fuente de verdad, en disco
                                      │
                        ┌─────────────┴─────────────┐
                        ▼                           ▼
                   lote.xlsx                   ItemDeCarga
              (revisión humana)                     │
                                                    ▼
                                        [ Cargador ]──→  formulario web
```

Entre etapa y etapa hay **un archivo en disco**, no una llamada de función. Eso
es lo que te deja cortar un lote de 3000 a la mitad y retomar, y lo que te deja
mirar el resultado intermedio cuando algo no cierra.

## Idea 1 · Las costuras

Lo primero que se decide no es la librería: es **dónde están las junturas**.

`contratos.py` no hace nada, y ese es el punto: define la forma que tienen que
tener las piezas para encajar.

In [ ]:
from pipeline.contratos import FichaCruda

ficha = FichaCruda(
    cabecera={"Cliente": "ACME SA", "CUIT": "30709204595"},
    tablas={"referencias": [{"Fecha": "12/5/2006", "Plazo": "30"}]},
    origen="ejemplo")

print(ficha.cabecera)
print(ficha.tabla("referencias"))

Fijate qué **no** hay ahí: ni fechas, ni números, ni CUIT validado. Todo strings.

**La regla de oro: el extractor no sabe de negocio.** No sabe qué es un CUIT ni
cómo se lee una antigüedad. Solo sabe abrir un archivo y poner texto en celdas.

Eso es lo que hace que se pueda reemplazar. El repo trae dos extractores —PDF y
DOCX— que por dentro no se parecen en nada y por fuera son intercambiables.

## Idea 2 · El ground truth

La idea que casi nadie aplica, y la que más te va a servir.

**El generador de datos falsos sabe exactamente qué escribió** en el archivo. Así
que puede dejar al lado, en un JSON, el resultado que un extractor correcto
tendría que devolver. Comparar contra eso es la diferencia entre *"parece que
anda"* y *"anda"*.

In [ ]:
correr(f"{PY} -m tests.verificar_extractor data/entrada/lote17.pdf | tail -4")

> Un parser puede leer 299 clientes perfecto y comerse una fila del trescientos.
> A ojo no lo ves nunca. **"Le pasé tres PDFs y anduvo" no es una verificación.**

## Idea 3 · Las defensas

Cargar en una web es la parte peligrosa: es donde escribís en un sistema que no
es tuyo. Tres defensas, en orden de importancia:

1. **Lista blanca.** Solo se escriben los campos declarados explícitamente. Uno
   fuera de la lista **aborta la fila** sin tocar la web.
2. **Nada más se movió.** Se fotografían *todos* los inputs del formulario antes
   y después de guardar. Si cambió algo fuera de la lista, es un error aunque el
   guardado haya "funcionado".
3. **Read-back.** Después de guardar se reabre el formulario y se compara. Que el
   submit no explote no significa que el dato entró.

Y una regla que las abarca a todas:

> **Ante la duda, no cargar.** Si dos filas del documento son indistinguibles, no
> se elige "la primera": se reportan y se dejan para un humano.

---

## Cómo sigue

| Cuaderno | Dónde corre | Qué hacés |
|---|---|---|
| **`02_construir`** | Colab hasta el Excel, después local | Escribís el pipeline sobre el caso de ejemplo |
| **`03_tu_caso`** | Tu máquina | Lo adaptás a tu archivo y tu web |

⚠️ **De la carga web en adelante, todo es local.** La máquina de Colab no llega a
tu web interna, y además vas a querer *ver* el navegador trabajando en tiempo
real, cosa que un cuaderno no puede mostrar.

Seguí con **`02_construir.ipynb`**.